# SST-2 fine-tuning: AdamW vs Muon (training)

Each entry in `configs` is one run. Every run reloads the model fresh, trains, and saves `ckpt/<name>/logs.json` plus weights at `save_steps` (and the step after each, for dW).

In [ ]:
import os, time, json
import numpy as np
import torch
from tqdm.auto import tqdm

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from torch.utils.data import DataLoader

name = "HuggingFaceTB/SmolLM2-135M"

tokenizer = AutoTokenizer.from_pretrained(name)
tokenizer.pad_token = tokenizer.eos_token          # tokenizer has no pad token; reuse eos (id 0)
tokenizer.padding_side = "right"                   # last-token pooling assumes right padding

dataset = load_dataset("stanfordnlp/sst2")

def tokenize(batch):
    return tokenizer(batch["sentence"], truncation=True, max_length=80)

tokenized = dataset.map(tokenize, batched=True)
tokenized = tokenized.remove_columns(["sentence", "idx"])
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch")

collator = DataCollatorWithPadding(tokenizer)
val_loader = DataLoader(tokenized["validation"], batch_size=128, shuffle=False, collate_fn=collator)


## Configs

In [14]:
base = dict(
    lr_adamw   = 2e-5,
    lr_muon    = 1e-3,
    momentum   = 0.95,
    wd         = 0.0,
    batch_size = 64,
    epochs     = 1,
    seed       = 1,
    eval_every = 100,
    save_steps = [0, 500, 1000],   # weights saved at these steps and the step after each; final step always saved
)


# All runs
configs = []

# main pair, 3 seeds
for seed in [1, 2, 3]:
    configs.append(dict(base, optimizer='adamw', seed=seed))
    configs.append(dict(base, optimizer='muon',  seed=seed))

# learning-rate sweep, both optimizers (the 5e-5 / 3e-4 pair also at seed 2)
configs += [
    dict(base, optimizer='adamw', lr_adamw=1e-5, seed=1),
    dict(base, optimizer='adamw', lr_adamw=5e-5, seed=1),
    dict(base, optimizer='adamw', lr_adamw=5e-5, seed=2),
    dict(base, optimizer='muon',  lr_muon=3e-4,  seed=1),
    dict(base, optimizer='muon',  lr_muon=3e-4,  seed=2),
    dict(base, optimizer='muon',  lr_muon=3e-3,  seed=1),
    dict(base, optimizer='muon',  lr_muon=3e-3,  seed=2),
]

# momentum ablation (Muon)
configs += [
    dict(base, optimizer='muon', momentum=0.0, seed=1),
    dict(base, optimizer='muon', momentum=0.9, seed=1),
]

# weight decay, same value for both optimizers
configs += [
    dict(base, optimizer='adamw', wd=0.01, seed=1),
    dict(base, optimizer='muon',  wd=0.01, seed=1),
]       

# config names that are used to save logs and checkpoints
def run_name(cfg):
    return '%s_lrA%g_lrM%g_mom%g_wd%g_bs%d_ep%d_seed%d' % (cfg['optimizer'], cfg['lr_adamw'], cfg['lr_muon'], cfg['momentum'],
                                                          cfg['wd'], cfg['batch_size'], cfg['epochs'], cfg['seed'])

[run_name(c) for c in configs]


['adamw_lrA2e-05_lrM0.001_mom0.95_wd0_bs64_ep1_seed1',
 'muon_lrA2e-05_lrM0.001_mom0.95_wd0_bs64_ep1_seed1',
 'adamw_lrA2e-05_lrM0.001_mom0.95_wd0_bs64_ep1_seed2',
 'muon_lrA2e-05_lrM0.001_mom0.95_wd0_bs64_ep1_seed2',
 'adamw_lrA2e-05_lrM0.001_mom0.95_wd0_bs64_ep1_seed3',
 'muon_lrA2e-05_lrM0.001_mom0.95_wd0_bs64_ep1_seed3',
 'adamw_lrA1e-05_lrM0.001_mom0.95_wd0_bs64_ep1_seed1',
 'adamw_lrA5e-05_lrM0.001_mom0.95_wd0_bs64_ep1_seed1',
 'adamw_lrA5e-05_lrM0.001_mom0.95_wd0_bs64_ep1_seed2',
 'muon_lrA2e-05_lrM0.0003_mom0.95_wd0_bs64_ep1_seed1',
 'muon_lrA2e-05_lrM0.0003_mom0.95_wd0_bs64_ep1_seed2',
 'muon_lrA2e-05_lrM0.003_mom0.95_wd0_bs64_ep1_seed1',
 'muon_lrA2e-05_lrM0.003_mom0.95_wd0_bs64_ep1_seed2',
 'muon_lrA2e-05_lrM0.001_mom0_wd0_bs64_ep1_seed1',
 'muon_lrA2e-05_lrM0.001_mom0.9_wd0_bs64_ep1_seed1',
 'adamw_lrA2e-05_lrM0.001_mom0.95_wd0.01_bs64_ep1_seed1',
 'muon_lrA2e-05_lrM0.001_mom0.95_wd0.01_bs64_ep1_seed1']

## Model, optimizer, eval

In [ ]:
def load_model():
    m = AutoModelForSequenceClassification.from_pretrained(name, num_labels=2, pad_token_id=tokenizer.pad_token_id, torch_dtype=torch.float32)
    return m.to(device)


def make_optimizers(model, cfg):
    params = list(model.named_parameters())
    muon_params  = [p for n, p in params if p.ndim == 2 and 'layers.' in n]
    other_params = [p for n, p in params if not (p.ndim == 2 and 'layers.' in n)]
    assert len(muon_params) == 210 and len(other_params) == 63

    if cfg['optimizer'] == 'muon':
        opts = [torch.optim.Muon(muon_params, lr=cfg['lr_muon'], momentum=cfg['momentum'], weight_decay=cfg['wd']),
                torch.optim.AdamW(other_params, lr=cfg['lr_adamw'], weight_decay=cfg['wd'])]
    else:
        opts = [torch.optim.AdamW(model.parameters(), lr=cfg['lr_adamw'], weight_decay=cfg['wd'])]
    return opts, muon_params, other_params


def evaluate(model, val_loader):
    val_loss = 0
    correct = 0
    total = 0
    model.eval()
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            outputs = model(**batch)
            val_loss += outputs.loss.item() * batch['labels'].size(0)
            preds = outputs.logits.argmax(dim=-1)
            correct += (preds == batch['labels']).sum().item()
            total += batch['labels'].size(0)
    model.train()
    return val_loss / total, correct / total


def grad_norm(params):

    norm = torch.norm(torch.stack([p.grad.norm() for p in params if p.grad is not None])).item()
    return norm


## Train Loop with logging and saving checkpoints and logs

### Muon acts on the 210 2-D weight matrices inside the transformer blocks (q/k/v/o, gate/up/down × 30 layers); the other 63 parameters (embedding, RMSNorm gains, classifier head) use AdamW

In [ ]:
def train(cfg):
    rname = run_name(cfg)
    out_dir = 'ckpt/' + rname

    if os.path.exists(out_dir + '/logs.json'):
        print('skip', rname)
        return json.load(open(out_dir + '/logs.json'))
        
    os.makedirs(out_dir, exist_ok=True)
    print('=' * 60)
    print(rname)

    torch.manual_seed(cfg['seed'])
    g = torch.Generator().manual_seed(cfg['seed'])   # same data order for both optimizers at a given seed
    train_loader = DataLoader(tokenized["train"], batch_size=cfg['batch_size'], shuffle=True, collate_fn=collator, generator=g)

    model = load_model()
    optimizers, muon_params, other_params = make_optimizers(model, cfg)

    save_at = set(cfg['save_steps']) | set(s + 1 for s in cfg['save_steps'])
    if 0 in save_at:
        torch.save(model.state_dict(), out_dir + '/step_0000.pt')

    train_loss, train_acc = [], []
    grad_norm_muon, grad_norm_other = [], []    # muon group = the 210 block matrices, in both runs
    val_log = []

    vl, va = evaluate(model, val_loader)
    val_log.append((0, vl, va))
    print('init   val_loss %.4f  val_acc %.4f' % (vl, va))

    step = 0
    t0 = time.time()
    pbar = tqdm(total=cfg['epochs'] * len(train_loader))
    model.train()
    for e in range(cfg['epochs']):
        for batch in train_loader:
            batch = batch.to(device)
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()

            train_loss.append(loss.item())
            train_acc.append((outputs.logits.argmax(-1) == batch['labels']).float().mean().item())
            grad_norm_muon.append(grad_norm(muon_params))
            grad_norm_other.append(grad_norm(other_params))

            for opt in optimizers:
                opt.step()
            for opt in optimizers:
                opt.zero_grad()

            step += 1
            pbar.update(1)
            pbar.set_postfix(loss='%.3f' % train_loss[-1], acc='%.2f' % train_acc[-1])

            if step % cfg['eval_every'] == 0:
                vl, va = evaluate(model, val_loader)
                val_log.append((step, vl, va))
                tqdm.write('step %4d  train_loss %.4f  val_loss %.4f  val_acc %.4f' % (step, np.mean(train_loss[-cfg['eval_every']:]), vl, va))

            if step in save_at:
                torch.save(model.state_dict(), out_dir + '/step_%04d.pt' % step)
    pbar.close()

    torch.save(model.state_dict(), out_dir + '/step_%04d.pt' % step)
    vl, va = evaluate(model, val_loader)
    val_log.append((step, vl, va))
    best = max(val_log, key=lambda x: x[2])

    print('final train loss (last 100): %.4f   train acc (last 100): %.4f' % (np.mean(train_loss[-100:]), np.mean(train_acc[-100:])))
    print('final val loss: %.4f   val acc: %.4f   best val acc: %.4f at step %d   time %.0fs' % (vl, va, best[2], best[0], time.time() - t0))

    logs = dict(cfg, run_name=rname, train_loss=train_loss, train_acc=train_acc,
                grad_norm_muon=grad_norm_muon, grad_norm_other=grad_norm_other, val_log=val_log, time=time.time() - t0)
    with open(out_dir + '/logs.json', 'w') as f:
        json.dump(logs, f)

    del model, optimizers
    if device == 'cuda':
       torch.cuda.empty_cache() #clear GPU memory

    return logs


# Run for all configs

In [12]:
results = {}
for cfg in configs:
    results[run_name(cfg)] = train(cfg)

skip adamw_lrA2e-05_lrM0.001_mom0.95_wd0_bs64_ep1_seed2
skip muon_lrA2e-05_lrM0.001_mom0.95_wd0_bs64_ep1_seed2
skip muon_lrA2e-05_lrM0.0003_mom0.95_wd0_bs64_ep1_seed1
skip muon_lrA2e-05_lrM0.003_mom0.95_wd0_bs64_ep1_seed1
skip muon_lrA2e-05_lrM0.001_mom0_wd0_bs64_ep1_seed1
skip adamw_lrA5e-05_lrM0.001_mom0.95_wd0_bs64_ep1_seed1
skip adamw_lrA1e-05_lrM0.001_mom0.95_wd0_bs64_ep1_seed1
skip muon_lrA2e-05_lrM0.001_mom0.95_wd0.01_bs64_ep1_seed1
skip adamw_lrA2e-05_lrM0.001_mom0.95_wd0.01_bs64_ep1_seed1
adamw_lrA2e-05_lrM0.001_mom0.95_wd0_bs64_ep1_seed3


Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1089.86it/s]
[transformers] LlamaForSequenceClassification LOAD REPORT from: HuggingFaceTB/SmolLM2-135M
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


init   val_loss 3.1035  val_acc 0.4908


  9%|▉         | 100/1053 [02:34<30:34,  1.92s/it, acc=0.84, loss=0.354]

step  100  train_loss 0.6300  val_loss 0.2700  val_acc 0.8979


...

step  900  train_loss 0.1832  val_loss 0.2462  val_acc 0.9094


 95%|█████████▍| 1000/1053 [51:19<02:15,  2.56s/it, acc=0.91, loss=0.238]

step 1000  train_loss 0.1911  val_loss 0.2718  val_acc 0.8888


100%|██████████| 1053/1053 [53:45<00:00,  3.06s/it, acc=0.95, loss=0.077]


final train loss (last 100): 0.1784   train acc (last 100): 0.9376
final val loss: 0.2682   val acc: 0.8933   best val acc: 0.9094 at step 800   time 3237s


# Print Results

In [7]:
print('%-60s %10s %10s %8s %10s' % ('run', 'final_acc', 'best_acc', 'best@', 'final_loss'))
for r, L in results.items():
    best = max(L['val_log'], key=lambda x: x[2])
    print('%-60s %10.4f %10.4f %8d %10.4f' % (r, L['val_log'][-1][2], best[2], best[0], L['val_log'][-1][1]))


run                                                           final_acc   best_acc    best@ final_loss
adamw_lrA2e-05_lrM0.001_mom0.95_wd0_bs64_ep1_seed2               0.9312     0.9323      900     0.2019
muon_lrA2e-05_lrM0.001_mom0.95_wd0_bs64_ep1_seed2                0.9140     0.9312      900     0.2078
muon_lrA2e-05_lrM0.0003_mom0.95_wd0_bs64_ep1_seed1               0.9266     0.9300      600     0.2062
muon_lrA2e-05_lrM0.003_mom0.95_wd0_bs64_ep1_seed1                0.8876     0.9094      300     0.2703
muon_lrA2e-05_lrM0.001_mom0_wd0_bs64_ep1_seed1                   0.9163     0.9186      400     0.3366
adamw_lrA5e-05_lrM0.001_mom0.95_wd0_bs64_ep1_seed1               0.9243     0.9289      600     0.1988
adamw_lrA1e-05_lrM0.001_mom0.95_wd0_bs64_ep1_seed1               0.9209     0.9232     1000     0.2249
muon_lrA2e-05_lrM0.001_mom0.95_wd0.01_bs64_ep1_seed1             0.9255     0.9358      700     0.1946
adamw_lrA2e-05_lrM0.001_mom0.95_wd0.01_bs64_ep1_seed1            0.9289  